# RU Index: перевод описаний + новый индекс + оценка

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.chdir(os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import json, numpy as np, faiss, time, re
from pathlib import Path
from tqdm import tqdm

with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_records = [json.loads(l) for l in f]

records = [r for r in all_records if not r.get("is_nsfw", False)]
print(f"Всего: {len(all_records)}, чистых: {len(records)}")


Всего: 9998, чистых: 9770


## 1. Перевод caption + ocr → RU (с сохранением прогресса)

In [2]:
from deep_translator import GoogleTranslator
import time

translator = GoogleTranslator(source="en", target="ru")

PROGRESS_FILE = Path("data/processed/ru_translations.json")

# Загружаем прогресс если есть
if PROGRESS_FILE.exists():
    with open(PROGRESS_FILE) as f:
        translations = json.load(f)
    print(f"Загружен прогресс: {len(translations)} переводов")
else:
    translations = {}
    print("Начинаем с нуля")

def safe_translate(text, max_len=400):
    """Переводит текст, возвращает оригинал при ошибке"""
    if not text or not text.strip():
        return ""
    text = text[:max_len]
    # Если текст уже русский (>50% кириллица) — не переводим
    cyr = sum(1 for c in text if "а" <= c.lower() <= "я" or c.lower() == "ё")
    if len(text) > 0 and cyr / len(text) > 0.3:
        return text
    try:
        result = translator.translate(text)
        return result or text
    except Exception as e:
        return text

# Переводим
errors = 0
for i, r in enumerate(tqdm(records, desc="Перевод")):
    fn = r["filename"]
    if fn in translations:
        continue

    caption = r.get("caption", "")
    main_idea = r.get("main_idea", "")
    ocr = r.get("ocr_text", "") or r.get("ocr_normalized", "")

    # Объединяем caption + main_idea в одну строку для перевода
    combined_en = " ".join(filter(None, [caption, main_idea]))[:400]

    cap_ru = safe_translate(combined_en)
    ocr_ru = safe_translate(ocr[:200]) if ocr else ""

    translations[fn] = {"caption_ru": cap_ru, "ocr_ru": ocr_ru}

    # Сохраняем прогресс каждые 100
    if (i + 1) % 100 == 0:
        with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
            json.dump(translations, f, ensure_ascii=False)
        time.sleep(0.5)  # пауза чтобы не забанили

# Финальное сохранение
with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
    json.dump(translations, f, ensure_ascii=False)

filled = sum(1 for v in translations.values() if v.get("caption_ru"))
print(f"Переведено: {filled}/{len(records)}")
print("Пример:")
for r in records[:3]:
    t = translations.get(r["filename"], {})
    print(f"  EN: {r.get('caption','')[:60]}")
    print(f"  RU: {t.get('caption_ru','')[:60]}")
    print()


Начинаем с нуля


Перевод: 100%|██████████| 9770/9770 [2:21:27<00:00,  1.15it/s]  

Переведено: 7413/9770
Пример:
  EN: I wish I was your math homework because then I'd be hard and
  RU: Мне бы хотелось, чтобы я был твоим домашним заданием по мате

  EN: A person with goat horns and a black robe is shown in a styl
  RU: На стилизованном плакате с надписью «САТАНИСТО4КА» изображен

  EN: An animated character is sitting at a table with a sign that
  RU: Анимационный персонаж сидит за столом с табличкой, на которо



## 2. Обновляем vqa_annotations_v2.jsonl — добавляем поля caption_ru, ocr_ru

In [3]:
updated = 0
with open("data/processed/vqa_annotations_v2.jsonl", "w", encoding="utf-8") as f:
    for r in all_records:
        t = translations.get(r.get("filename", ""), {})
        r["caption_ru"] = t.get("caption_ru", "")
        r["ocr_ru"] = t.get("ocr_ru", "")
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
        if t.get("caption_ru"):
            updated += 1

print(f"Обновлено записей: {updated}/{len(all_records)}")
print("Поля caption_ru и ocr_ru добавлены в vqa_annotations_v2.jsonl")


Обновлено записей: 9725/9998
Поля caption_ru и ocr_ru добавлены в vqa_annotations_v2.jsonl


## 3. Строим RU текстовый индекс (emb_caption_ru)

In [5]:
from sentence_transformers import SentenceTransformer


model = SentenceTransformer("BAAI/bge-m3", device="cpu")


# Формируем RU тексты для индекса
ru_texts = []
for r in records:
    parts = []
    cap_ru = r.get("caption_ru", "").strip()
    ocr_ru = r.get("ocr_ru", "").strip()
    if cap_ru:
        parts.append(cap_ru)
    if ocr_ru and ocr_ru not in cap_ru:
        parts.append(ocr_ru)
    ru_texts.append(" ".join(parts) if parts else "")

non_empty = sum(1 for t in ru_texts if t)
print(f"RU текстов: {non_empty}/{len(ru_texts)} непустых")
print(f"Пример: {ru_texts[0][:80]}")

print("Encode RU текстов...")
safe_ru = [t if t.strip() else "пусто" for t in ru_texts]
emb_ru = model.encode(safe_ru, normalize_embeddings=True, batch_size=32,
                      show_progress_bar=True).astype(np.float32)

# Зануляем пустые
for i, t in enumerate(ru_texts):
    if not t.strip():
        emb_ru[i] = np.zeros(emb_ru.shape[1])

print(f"emb_ru: {emb_ru.shape}")

# Сохраняем
np.save("data/processed/emb_caption_ru.npy", emb_ru)
idx_ru = faiss.IndexFlatIP(emb_ru.shape[1])
idx_ru.add(emb_ru)
faiss.write_index(idx_ru, "data/processed/faiss_caption_ru.index")
print(f"Сохранено: faiss_caption_ru.index ({idx_ru.ntotal} vectors)")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

RU текстов: 9745/9770 непустых
Пример: Мне бы хотелось, чтобы я был твоим домашним заданием по математике, потому что т
Encode RU текстов...


Batches:   0%|          | 0/306 [00:00<?, ?it/s]

emb_ru: (9770, 1024)
Сохранено: faiss_caption_ru.index (9770 vectors)


## 4. Оценка: пайплайн с RU индексом vs без

In [6]:
import re
from rank_bm25 import BM25Okapi

# Загружаем данные
with open("data/processed/index_metadata.jsonl") as f:
    metadata = [json.loads(l) for l in f]

idx_caption = faiss.read_index("data/processed/faiss_caption.index")
idx_ocr = faiss.read_index("data/processed/faiss_ocr.index")
idx_keywords = faiss.read_index("data/processed/faiss_keywords.index")
idx_caption_ru = faiss.read_index("data/processed/faiss_caption_ru.index")

# Обновляем метаданные с RU полями
with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_rec = [json.loads(l) for l in f]
fn_to_ru = {r["filename"]: {"caption_ru": r.get("caption_ru",""), "ocr_ru": r.get("ocr_ru","")} for r in all_rec}
for m in metadata:
    ru = fn_to_ru.get(m["filename"], {})
    m["caption_ru"] = ru.get("caption_ru", "")
    m["ocr_ru"] = ru.get("ocr_ru", "")

def tokenize(t): return re.findall(r"[a-zа-яё0-9]+", t.lower())

# BM25 по EN+RU текстам
corpus = []
for m in metadata:
    parts = [m.get("caption",""), m.get("ocr_text",""), m.get("caption_ru",""), m.get("ocr_ru","")]
    corpus.append(tokenize(" ".join(parts)))
bm25 = BM25Okapi(corpus)
print("BM25 (EN+RU) готов")

# Маппинг индексов
with open("data/processed/vqa_annotations_v2.jsonl") as f:
    all_recs = [json.loads(l) for l in f]
o2n = {}; ni = 0
for oi, r in enumerate(all_recs):
    if not r.get("is_nsfw"): o2n[oi]=ni; ni+=1

with open("eval/language_experiment_queries.json") as f:
    queries = json.load(f)
valid = []
for q in queries:
    nn = o2n.get(q["index"])
    if nn is not None:
        qc = dict(q); qc["index"] = nn; valid.append(qc)
print(f"Запросов: {len(valid)}")


BM25 (EN+RU) готов
Запросов: 100


In [7]:
from deep_translator import GoogleTranslator
translator_q = GoogleTranslator(source="ru", target="en")

def translate_q(text):
    cyr = sum(1 for c in text if "а" <= c.lower() <= "я")
    if len(text) > 0 and cyr / len(text) < 0.3:
        return text
    try: return translator_q.translate(text) or text
    except: return text

# Pre-encode запросов
ru_texts_q = [q.get("query_ru","").strip() or q["caption"][:100] for q in valid]
en_texts_q = [q.get("query_en","").strip() or q["caption"][:100] for q in valid]


ru_embs = model.encode(ru_texts_q, normalize_embeddings=True, batch_size=16).astype(np.float32)

en_embs = model.encode(en_texts_q, normalize_embeddings=True, batch_size=16).astype(np.float32)

ru_tr_texts = [translate_q(t) for t in ru_texts_q]
ru_tr_embs = model.encode(ru_tr_texts, normalize_embeddings=True, batch_size=16).astype(np.float32)



Encode RU запросов...
Encode EN запросов...
Translate + encode RU→EN...
Готово


In [8]:
def rrf(rls, k=60):
    s={}
    for rl in rls:
        for r,d in enumerate(rl): s[d]=s.get(d,0)+1.0/(k+r+1)
    return sorted(s, key=lambda x:-s[x])

def evaluate(embs, lang_key, use_bm25=False, use_multi=False,
             use_ru_idx=False, tr_embs=None):
    h1=h5=h10=0; mrr=0
    for i, q in enumerate(valid):
        rl = []
        qe = embs[i:i+1]

        _, ci = idx_caption.search(qe, 50); rl.append(ci[0].tolist())
        if use_multi:
            _, oi = idx_ocr.search(qe, 50); rl.append(oi[0].tolist())

        if use_bm25:
            qt = q.get(lang_key,"").strip() or q["caption"][:100]
            rl.append(np.argsort(-bm25.get_scores(tokenize(qt)))[:50].tolist())

        # RU индекс
        if use_ru_idx:
            _, ri = idx_caption_ru.search(qe, 50); rl.append(ri[0].tolist())

        # Перевод запроса → EN
        if tr_embs is not None:
            te = tr_embs[i:i+1]
            _, tci = idx_caption.search(te, 50); rl.append(tci[0].tolist())
            if use_bm25:
                qt_tr = ru_tr_texts[i]
                rl.append(np.argsort(-bm25.get_scores(tokenize(qt_tr)))[:50].tolist())

        ranking = rrf(rl)
        t = q["index"]
        if t in ranking:
            p = ranking.index(t)+1
            if p<=1:h1+=1
            if p<=5:h5+=1
            if p<=10:h10+=1
            if p<=10:mrr+=1.0/p
    n=len(valid)
    return h1/n,h5/n,h10/n,mrr/n

print(f"{'Конфигурация':<45} {'Hit@1':>6} {'Hit@5':>6} {'Hit@10':>7} {'MRR':>8}")


configs = [
    ("EN: Caption only (baseline)", en_embs, "query_en", {}),
    ("EN: Multi+BM25", en_embs, "query_en", {"use_bm25":True,"use_multi":True}),
    ("RU: Caption only", ru_embs, "query_ru", {}),
    ("RU: Multi+BM25", ru_embs, "query_ru", {"use_bm25":True,"use_multi":True}),
    ("RU: +RU индекс", ru_embs, "query_ru", {"use_bm25":True,"use_multi":True,"use_ru_idx":True}),
    ("RU: +translate query", ru_embs, "query_ru", {"use_bm25":True,"use_multi":True,"tr_embs":ru_tr_embs}),
    ("RU: +RU индекс +translate", ru_embs, "query_ru", {"use_bm25":True,"use_multi":True,"use_ru_idx":True,"tr_embs":ru_tr_embs}),
]

for name, embs, lang, kw in configs:
    a,b,c,d = evaluate(embs, lang, **kw)
    print(f"{name:<45} {a:>6.0%} {b:>6.0%} {c:>7.0%} {d:>8.4f}")

print()
print("Прошлые результаты:")
print("  Multi+BM25+CLIP (EN):         Hit@5=54%")
print("  Multi+BM25+CLIP+Translate RU: Hit@5=44%")


Конфигурация                                   Hit@1  Hit@5  Hit@10      MRR
EN: Caption only (baseline)                      23%    39%     44%   0.2968
EN: Multi+BM25                                   19%    41%     47%   0.2843
RU: Caption only                                 14%    37%     43%   0.2341
RU: Multi+BM25                                   23%    42%     48%   0.3081
RU: +RU индекс                                   24%    46%     57%   0.3385
RU: +translate query                             21%    42%     53%   0.3121
RU: +RU индекс +translate                        22%    45%     55%   0.3291

Прошлые результаты:
  Multi+BM25+CLIP (EN):         Hit@5=54%
  Multi+BM25+CLIP+Translate RU: Hit@5=44%


clip на ru не дает прироста, только на английском, тк как работает только на eng